# Build the IL2CPP Dumper Studio APK on Google Colab
Developed by Mohamed Annati.

Run the two cells in order. Cell 1 installs JDK 17 + Android SDK + Gradle (skipped automatically if already present) and builds the release APK. Cell 2 downloads the APK.

The APK uses the same hexagon + `</>` icon as the web studio and the same on-device dump features (dump.cs / stringliteral.json / il2cpp.h).

Notes: `curl -fsSL` is required (Gradle's URL redirects). The `r2u ... Sources` apt warning is harmless. Re-running is fast: SDK/Gradle are reused and only the source is re-cloned.

In [ ]:
%%bash
set -e
cd /content

# 1) JDK 17 (skip if java already present)
if ! command -v java >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq openjdk-17-jdk >/dev/null
fi
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

# 2) latest source (includes the Kotlin compile fixes + new icon)
rm -rf mainproject
git clone -q -b arena/01a0501d-mainproject https://github.com/Mohamed2020p/mainproject

# 3) Android SDK (skip if already installed)
export ANDROID_HOME=/content/sdk ANDROID_SDK_ROOT=/content/sdk
if [ ! -x sdk/cmdline-tools/latest/bin/sdkmanager ]; then
  mkdir -p sdk/cmdline-tools
  curl -fsSL -o ct.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
  unzip -q ct.zip -d sdk/cmdline-tools
  mv sdk/cmdline-tools/cmdline-tools sdk/cmdline-tools/latest
  yes | sdk/cmdline-tools/latest/bin/sdkmanager --sdk_root=/content/sdk \
      "platforms;android-34" "build-tools;34.0.0" "platform-tools" >/dev/null
fi

# 4) Gradle 8.7 (skip if already present)  -L follows redirects, -f fails on HTTP errors
if [ ! -x gradle-8.7/bin/gradle ]; then
  curl -fsSL -o gradle.zip https://services.gradle.org/distributions/gradle-8.7-bin.zip
  unzip -q gradle.zip
fi
export PATH=/content/gradle-8.7/bin:$PATH

# 5) build
cd mainproject/android
echo "sdk.dir=/content/sdk" > local.properties
gradle --no-daemon assembleRelease

cp app/build/outputs/apk/release/app-release-unsigned.apk /content/IL2CPPDumperStudio.apk
echo "[+] APK ready: /content/IL2CPPDumperStudio.apk"

In [ ]:
from google.colab import files
files.download('/content/IL2CPPDumperStudio.apk')